# SSVEP Responses

## Overview
This notebook analyzes SSVEP responses from Nakanishi2015. It computes PSD at occipital electrodes (O1, Oz, O2) for four stimulus frequencies (9.25, 9.75, 11.25, 13.25 Hz).

## What to look for
- Peaks in spectrum at stimulus frequency and harmonics
- High SNR at stimulus frequency

## 1. Install dependencies

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn

## 2. Load data

In [ ]:
import numpy as np
from scipy.signal import welch
from moabb.datasets import Nakanishi2015
from moabb.paradigms import SSVEP

FS = 256
dataset = Nakanishi2015()
paradigm = SSVEP(fmin=7, fmax=45, n_classes=4)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

raw = dataset.get_data(subjects=[1])
s1 = raw[1]; sess = list(s1.values())[0]; run = list(sess.values())[0]
ch_names = run.ch_names
target_chs = ['O1', 'Oz', 'O2']
ch_idx = [ch_names.index(c) for c in target_chs if c in ch_names]
target_chs = [ch_names[i] for i in ch_idx]
print(f"Data: {X.shape}, Frequencies: {np.unique(labels)}")

## 3. Interactive plot

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

unique_labels = np.unique(labels)
fig = make_subplots(rows=2, cols=2, subplot_titles=[f'{l} Hz' for l in unique_labels])
for i, label in enumerate(unique_labels):
    epochs = X[labels == label]
    avg = epochs.mean(axis=0)
    row = i // 2 + 1; col = i % 2 + 1
    for ci, cn in zip(ch_idx, target_chs):
        freqs, psd = welch(avg[ci, :], fs=FS, nperseg=min(1024, avg.shape[1]))
        stim = float(label)
        fig.add_trace(go.Scatter(x=freqs, y=np.log10(psd+1e-12), name=cn, mode='lines', showlegend=(i==0)), row=row, col=col)
        fig.add_vline(x=stim, line_dash='dash', line_color='red', row=row, col=col)
    fig.update_xaxes(range=[5, 50], row=row, col=col)
fig.update_layout(title='SSVEP PSD - Nakanishi2015 Subject 1', width=1100, height=700)
fig.show()

## Summary
- SSVEP is a response at visual stimulus frequency + harmonics
- Appears at occipital electrodes (O1, Oz, O2)
- Enables fast communication (100+ words/min)